# Assignment 5, Question 6: Data Transformation

**Points: 20**

Transform and engineer features from the clinical trial dataset.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import utilities
from q3_data_utils import load_data, clean_data, transform_types, create_bins, fill_missing

df = load_data('data/clinical_trial_raw.csv')
print(f"Loaded {len(df
)} patients")

# Prewritten visualization functions for transformation analysis
def plot_distribution(series, title, figsize=(10, 6)):
    """
    Create a histogram of a numeric series.
    
    Args:
        series: pandas Series with numeric data
        title: Chart title
        figsize: Figure size tuple
    """
    plt.figure(figsize=figsize)
    series.hist(bins=30)
    plt.title(title)
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

def plot_value_counts(series, title, figsize=(10, 6)):
    """
    Create a bar chart of value counts.
    
    Args:
        series: pandas Series with value counts
        title: Chart title
        figsize: Figure size tuple
    """
    plt.figure(figsize=figsize)
    series.plot(kind='bar')
    plt.title(title)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

Loaded 10000 patients


## Part 1: Type Conversions (5 points)

1. Convert 'enrollment_date' to datetime using the `transform_types()` utility
2. Convert categorical columns ('site', 'intervention_group', 'sex') to category dtype
3. Ensure all numeric columns are proper numeric types
4. Display the updated dtypes

In [40]:
# TODO: Type conversions
# 1. Use transform_types() to convert enrollment_date to datetime
# 2. Convert categorical columns ('site', 'intervention_group', 'sex') to category dtype
# 3. Ensure all numeric columns are proper numeric types
# 4. Display the updated dtypes using df.dtypes


type_map = {
    'enrollment_date': 'datetime',
    'site': 'category',
    'intervention_group': 'category',
    'sex': 'category',
    'age': 'numeric',
    'bmi': 'numeric',
    'systolic_bp': 'numeric',
    'diastolic_bp': 'numeric',
    'cholesterol_total': 'numeric',
    'cholesterol_hdl': 'numeric',
    'cholesterol_ldl': 'numeric',
    'glucose_fasting': 'numeric',
    'follow_up_months': 'numeric',
    'adverse_events': 'numeric',
    'adherence_pct': 'numeric'
}
df = transform_types(df, type_map)

print(df.dtypes)


patient_id                    object
age                            int64
sex                         category
bmi                          float64
enrollment_date       datetime64[ns]
systolic_bp                  float64
diastolic_bp                 float64
cholesterol_total            float64
cholesterol_hdl              float64
cholesterol_ldl              float64
glucose_fasting              float64
site                        category
intervention_group          category
follow_up_months               int64
adverse_events                 int64
outcome_cvd                   object
adherence_pct                float64
dropout                       object
dtype: object


## Part 2: Feature Engineering (8 points)

Create these new calculated columns:

1. `cholesterol_ratio` = cholesterol_ldl / cholesterol_hdl
2. `bp_category` = categorize systolic BP:
   - 'Normal': < 120
   - 'Elevated': 120-129
   - 'High': >= 130
3. `age_group` using `create_bins()` utility:
   - Bins: [0, 40, 55, 70, 100]
   - Labels: ['<40', '40-54', '55-69', '70+']
4. `bmi_category` using standard BMI categories:
   - Underweight: <18.5
   - Normal: 18.5-24.9
   - Overweight: 25-29.9
   - Obese: >=30

In [41]:
# TODO: Calculate cholesterol ratio
df['cholesterol_ratio'] = df['cholesterol_ldl'] / df['cholesterol_hdl']
print(df[['cholesterol_ldl', 'cholesterol_hdl', 'cholesterol_ratio']].head())

   cholesterol_ldl  cholesterol_hdl  cholesterol_ratio
0             41.0             55.0           0.745455
1            107.0             58.0           1.844828
2             82.0             56.0           1.464286
3            104.0             56.0           1.857143
4             75.0             78.0           0.961538


In [42]:
# TODO: Categorize blood pressure
def categorize_bp(sbp):
    if sbp < 120:
        return 'Normal'
    elif 120 <= sbp <= 129:
        return 'Elevated'
    else:
        return 'High'

df['bp_category'] = df['systolic_bp'].apply(categorize_bp)
print(df[['systolic_bp', 'bp_category']].head())

   systolic_bp bp_category
0        123.0    Elevated
1        139.0        High
2        123.0    Elevated
3        116.0      Normal
4         97.0      Normal


**Note:** The `create_bins()` function has an optional `new_column` parameter. If you don't specify it, the new column will be named `{original_column}_binned`. You can use `new_column='age_group'` to give it a custom name.


In [43]:
# TODO: Create age groups
age_bins = [0, 40, 55, 70, 100]
age_labels = ['<40', '40-54', '55-69', '70+']
df = create_bins(df, column='age', bins=age_bins, labels=age_labels, new_column='age_group')
print(df[['cholesterol_ratio', 'bp_category', 'age_group']].head())

   cholesterol_ratio bp_category age_group
0           0.745455    Elevated       70+
1           1.844828        High       70+
2           1.464286    Elevated       70+
3           1.857143      Normal       70+
4           0.961538      Normal       70+


In [44]:
# TODO: Create BMI categories
df['bmi'] = df['bmi'].apply(lambda x: x if x > 0 else np.nan)

bmi_bins = [0, 18.5, 24.9, 29.9, np.inf]
bmi_labels = ['Underweight', 'Normal', 'Overweight', 'Obese']
df = create_bins(df, column='bmi', bins=bmi_bins, labels=bmi_labels, new_column='bmi_category')
print(df[['bmi', 'bmi_category']].head())   

    bmi bmi_category
0  29.3   Overweight
1   NaN          NaN
2   NaN          NaN
3  25.4   Overweight
4   NaN          NaN


## Part 3: String Cleaning (2 points)

If there are any string columns that need cleaning:
1. Convert to lowercase
2. Strip whitespace
3. Replace any placeholder values

In [45]:
# TODO: String cleaning

string_cols = df.select_dtypes(include='object').columns

for col in string_cols:
    df[col] = df[col].str.lower()        
    df[col] = df[col].str.strip()      
    df[col] = df[col].replace({'na': np.nan, 'n/a': np.nan, 'unknown': np.nan})  

df[string_cols].head()

,patient_id,outcome_cvd,dropout,bp_category
0,p00001,no,no,elevated
1,p00002,no,no,high
2,p00003,yes,no,elevated
3,p00004,no,no,normal
4,p00005,yes,yes,normal


## Part 4: One-Hot Encoding (5 points)

Create dummy variables for categorical columns:
1. One-hot encode 'intervention_group' using `pd.get_dummies()`
2. One-hot encode 'site'
3. Drop the original categorical columns
4. Show the new shape and column names

In [46]:
# TODO: One-hot encoding

categorical_cols = ['intervention_group', 'site']

for col in categorical_cols:
    dummies = pd.get_dummies(df[col], prefix=col)
    df = pd.concat([df, dummies], axis=1)
    df.drop(columns=[col], inplace=True)

print("New dataset shape:", df.shape)
print("Columns after one-hot encoding:", df.columns.tolist())


New dataset shape: (10000, 80)
Columns after one-hot encoding: ['patient_id', 'age', 'sex', 'bmi', 'enrollment_date', 'systolic_bp', 'diastolic_bp', 'cholesterol_total', 'cholesterol_hdl', 'cholesterol_ldl', 'glucose_fasting', 'follow_up_months', 'adverse_events', 'outcome_cvd', 'adherence_pct', 'dropout', 'cholesterol_ratio', 'bp_category', 'age_group', 'bmi_category', 'intervention_group_  CONTROL  ', 'intervention_group_  Contrl  ', 'intervention_group_  Control  ', 'intervention_group_  TREATMENT A  ', 'intervention_group_  TREATMENT B  ', 'intervention_group_  Treatmen A  ', 'intervention_group_  Treatment  B  ', 'intervention_group_  Treatment A  ', 'intervention_group_  Treatment B  ', 'intervention_group_  TreatmentA  ', 'intervention_group_  control  ', 'intervention_group_  treatment a  ', 'intervention_group_  treatment b  ', 'intervention_group_CONTROL', 'intervention_group_Contrl', 'intervention_group_Control', 'intervention_group_TREATMENT A', 'intervention_group_TREATMEN

## Part 5: Save Transformed Data

Save the fully transformed dataset to `output/q6_transformed_data.csv`

In [47]:
# TODO: Save transformed data
# df_transformed.to_csv('output/q6_transformed_data.csv', index=False)

import os

# Ensure the output directory exists
os.makedirs('output', exist_ok=True)

# Save the transformed DataFrame
df.to_csv('output/q6_transformed_data.csv', index=False)

print("Transformed dataset saved to output/q6_transformed_data.csv")


Transformed dataset saved to output/q6_transformed_data.csv
